# 02 – Analysis

How much did knife prices fall after the supply shock of October 2025, and what explains the differences between knives?

The dependent variable is the log change of each knife's price between 22 and 24 October 2025. The explanatory variables describe the market of the knife (concentration, liquidity, share of western marketplaces) and the supply that could be converted into it: the stock and value of the covert guns and cases that lead to the same knife.

Input: `final.csv` from `01_data_collection.ipynb`.

In [ ]:
import ast
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from tqdm import tqdm

## 1. Cleaning

In [ ]:
df = pd.read_csv("final.csv") #Final dataset of the full run
df = df.loc[df["stpr"]>0] #Keep items with a positive price on 22 October
df = df.drop(columns=["API","stock","price"]) #Drop the listing-page price and stock, the API values are used instead
df["caseid"] = df["caseid"].apply(ast.literal_eval) #Case links: from string to list
df = df.loc[df["Souvenir"]==False] #Drop Souvenir items
df = df.drop(columns=["Souvenir"])
df["stattrak"] = df["stattrak"] == True #StatTrak flag as boolean
df

In [ ]:
knifes = df.loc[df["case"]=="knife"].copy() #Knives
knifes = knifes.loc[knifes["enpr"]>0] #Keep knives with a positive price on 24 October
knifes["diff"] = np.log(knifes["enpr"]) - np.log(knifes["stpr"]) #Dependent variable: log price change between the two dates
knifes = knifes.reset_index(drop=True)
knifes

## 2. Convertible supply

For each knife: the stock and value of the covert guns (with the same StatTrak status) and of the cases that come from the same cases as the knife.

In [ ]:
guns = df.loc[df["case"]=="gun"][["stattrak","caseid","stst","stpr"]].copy() #Covert guns
guns["val"] = guns["stst"]*guns["stpr"] #Total value of the gun's stock
guns = guns[["caseid","stattrak","stst","val"]]
guns = guns.reset_index(drop=True)

joiner = [] #Convertible gun stock and value per knife

for I in tqdm(range(len(knifes))): #Iterate through the knives
    STB = bool(knifes["stattrak"][I]) #StatTrak status of the knife
    subgun = guns.loc[[bool(set(knifes["caseid"][I]) & set(gun)) for gun in guns["caseid"]]] #Guns from at least one of the knife's cases
    subgun = subgun.loc[subgun["stattrak"]==STB] #with the same StatTrak status
    totstock = sum(subgun["stst"]) #Their total stock
    totval = sum(subgun["val"]) #Their total value
    joiner.append([str(knifes["url"][I]),totstock,totval])

joiner = pd.DataFrame(joiner,columns=["url","gun_stock","gun_val"])
analysis = knifes.join(joiner.set_index("url"),how="left",on="url") #Attach to the knives
analysis

In [ ]:
case = df.loc[df["case"]=="case"][["url","stpr","stst"]].copy() #Cases
case = case.reset_index(drop=True)
case["val"] = case["stpr"]*case["stst"] #Total value of the case's stock
case = case.drop(columns=["stpr"])
case["url"] = case["url"] + "-skins" #Rewrite the case URL to the format used in the knives' case links
case["url"] =[urlurl.replace("/cs2-items/container","/cs2-skin-search/skin-cases") for urlurl in case["url"]]

joiner = [] #Convertible case stock and value per knife

for I in tqdm(range(len(knifes))): #Iterate through the knives
    subcase = case.loc[[str(cas) in knifes["caseid"][I] for cas in case["url"]]] #Cases the knife can be obtained from
    totval = sum(subcase["val"]) #Their total value
    totstock = sum(subcase["stst"]) #Their total stock
    joiner.append([str(knifes["url"][I]),totstock,totval])

joiner = pd.DataFrame(joiner,columns=["url","case_stock","case_val"])
analysis2 = analysis.join(joiner.set_index("url"),how="left",on="url") #Attach to the knives
analysis2

In [ ]:
analysis3 = analysis2.copy()
analysis3["gun_relstock"] = analysis3["gun_stock"]/analysis3["stst"] #Convertible gun stock relative to the knife's own stock
analysis3["gun_meanval"] = analysis3["gun_val"]/analysis3["gun_stock"] #Mean value of the convertible guns
analysis3["gun_relval"] = analysis3["gun_meanval"]/analysis3["stpr"] #relative to the knife's price

analysis3["case_relstock"] = analysis3["case_stock"]/analysis3["stst"] #Case stock relative to the knife's own stock
analysis3["case_meanval"] = analysis3["case_val"]/analysis3["case_stock"] #Mean value of the cases
analysis3["case_relval"] = analysis3["case_meanval"]/analysis3["stpr"] #relative to the knife's price

analysis3 = analysis3.loc[analysis3["stst"]>100] #Keep knives with more than 100 items in stock on 22 October

regdf = analysis3[["diff","west","HHI","liq","gun_relstock","gun_relval","case_relstock","case_relval"]].copy() #Regression variables
regdf = regdf.dropna()

for var in regdf.columns[1:]:
    regdf = regdf.loc[regdf[var]>=0] #Drop negative values, which can only come from data errors

regdf["west"] = regdf["west"]/100 #Share of western marketplaces, 0-1
regdf["liq"] = regdf["liq"]/100 #Liquidity, 0-1

regdf

## 3. Functional form

In [ ]:
expl_vars = regdf.columns[1:].tolist() #Candidate explanatory variables

y = regdf["diff"] #Dependent variable
X = regdf[expl_vars].copy()
X["const"] = 1

spec_model = sm.OLS(y, X).fit() #Linear baseline

sm.stats.diagnostic.linear_reset(spec_model) #Ramsey RESET test
#p < 0.05: the linear form is misspecified

In [ ]:
for var in expl_vars: #Scatter plot of each variable against the price change, in levels and in logs
    x = X[var]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(x, y, alpha=0.6)
    coef1 = np.polyfit(x, y, 2) #Quadratic trend line
    poly1 = np.poly1d(coef1)
    xx1 = np.linspace(x.min(), x.max(), 300)
    axes[0].plot(xx1, poly1(xx1), color='red', linewidth=2)
    axes[0].set_title(f"{var} and the price change")

    coef2 = np.polyfit(np.log(x), y, 2) #Quadratic trend line on the log scale
    poly2 = np.poly1d(coef2)
    xx2 = np.linspace(np.log(x).min(), np.log(x).max(), 300)
    axes[1].scatter(np.log(x), y, alpha=0.6)
    axes[1].plot(xx2, poly2(xx2), color='red', linewidth=2)
    axes[1].set_title(f"log({var}) and the price change")

plt.show()

In [ ]:
regdf["west2"] = regdf["west"]**2 #Square of the western share
regdf["sqrt_gun_relval"] = np.sqrt(regdf["gun_relval"]) #Square root of the relative gun value
regdf["log_case_relstock"] = np.log(regdf["case_relstock"]) #Log of the relative case stock
regdf["log_case_relval"] = np.log(regdf["case_relval"]) #Log of the relative case value

regdf = regdf.drop(columns=["west","gun_relval","case_relstock","case_relval"]) #Keep the transformed versions
main_effects = regdf.columns[1:].tolist() #The seven main effects
regdf

## 4. Interaction screening

Exploratory step: every pairwise product of the main effects is ranked by its correlation with the price change, and the five strongest are added to the model. Because the interactions are chosen by looking at the outcome, the p-values of the later models are not valid for formal inference.

In [ ]:
pairs = list(itertools.combinations(main_effects, 2)) #All pairs of main effects

intercor = [] #Correlation of each product with the price change

for pair in pairs:
    intercor.append([pair[0]+"_x_"+pair[1], np.corrcoef(regdf[pair[0]]*regdf[pair[1]],regdf["diff"])[0][1]])

intercor = pd.DataFrame(intercor,columns=["interaction","corcoef"])
intercor["corcoef"] = np.abs(intercor["corcoef"]) #Strength of the correlation
intercor = intercor.sort_values("corcoef",ascending=False)
intercor = intercor[0:5] #Five strongest
intercor

In [ ]:
for Int in intercor["interaction"]: #Add the five interactions to the data
    comp = str(Int).split("_x_")
    regdf[str(Int)] = regdf[comp[0]]*regdf[comp[1]]

regdf

## 5. Models

In [ ]:
y = regdf["diff"]

X1 = regdf[main_effects].copy() #Model 1: main effects only
X1["const"] = 1
model1 = sm.OLS(y, X1).fit()
print(model1.summary())

In [ ]:
X2 = regdf[regdf.columns[1:]].copy() #Model 2: main effects and the five interactions
X2["const"] = 1
model2 = sm.OLS(y, X2).fit()
print(model2.summary())

print(sm.stats.diagnostic.het_white(model2.resid,X2)) #White test for heteroscedasticity
#p < 0.05: heteroscedastic errors, so robust standard errors are used below

model2_r = sm.OLS(y, X2).fit(cov_type='HC0') #Model 2 with heteroscedasticity-robust (HC0) standard errors
print(model2_r.summary())

In [ ]:
keep = [var for var in model2_r.pvalues.index if model2_r.pvalues[var]<0.05 and var!="const"] #Terms significant at 5% in robust model 2

X3 = regdf[keep].copy() #Model 3: only those terms
X3["const"] = 1
model3 = sm.OLS(y, X3).fit(cov_type='HC0') #with robust standard errors
print(model3.summary())

## 6. Model comparison

The F-test of nested models assumes the classical covariance matrix, so it is run on the non-robust fits.

In [ ]:
def WALDFTEST(full,restr): #F-test of a restricted model against a fuller one
    F_stat, p_value, df_diff = full.compare_f_test(restr)
    print("F-statistic:", F_stat)
    print("p-value:", p_value)
    print("df_diff:", df_diff)

model3_ols = sm.OLS(y, X3).fit() #Non-robust fit of model 3, for the F-test

WALDFTEST(model2,model1) #Model 1 vs model 2: do the interactions add explanatory power?
#p < 0.05: the interactions improve the fit, model 2 is preferred to model 1

WALDFTEST(model2,model3_ols) #Model 3 vs model 2: do the dropped terms matter?
#p > 0.05: the dropped terms add nothing significant, model 3 is preferred as it uses fewer parameters

In [ ]:
sm.stats.diagnostic.linear_reset(model3,power=3) #RESET test of the final model
#Still p < 0.05, but the statistic is lower than for the linear baseline: the specification improved but is not fully correct